# Submit Azure ML Pipelines with the Python SDK

This notebook loads an existing YAML job, overrides its compute target, and submits it with `azure-ai-ml`.

Configure the repository-root `.env` first. Submission is disabled unless `RUN_AZUREML_JOB=true`. Authenticate separately with `az login` locally or `az login --identity` on an Azure ML compute instance.

## `.env` settings

Add these values to the repository-root `.env`:

```dotenv
AZURE_SUBSCRIPTION_ID=<SUBSCRIPTION_ID>
AZURE_TENANT_ID=<TENANT_ID>
AZURE_RESOURCE_GROUP=<RESOURCE_GROUP_NAME>
AZUREML_WORKSPACE_NAME=<AZURE_ML_WORKSPACE_NAME>
AZUREML_COMPUTE_NAME=<COMPUTE_NAME>
AZUREML_PIPELINE_FILE=pipelines/single-step-merge-job.yaml
RUN_AZUREML_JOB=false
```

To run the integration pipeline instead, set `AZUREML_PIPELINE_FILE=pipelines/integration-compare-pipeline.yaml`.

In [ ]:
from pathlib import Path
import os

from azure.ai.ml import MLClient, load_job
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (folder / ".env.example").is_file() and (folder / "pipelines").is_dir():
        REPO_ROOT = folder
        break
else:
    raise FileNotFoundError("Run this notebook from inside the repository")

load_dotenv(REPO_ROOT / ".env")

SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID", "").strip()
TENANT_ID = os.getenv("AZURE_TENANT_ID", "").strip()
RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "").strip()
WORKSPACE_NAME = os.getenv("AZUREML_WORKSPACE_NAME", "").strip()
COMPUTE_NAME = os.getenv("AZUREML_COMPUTE_NAME", "").strip()
PIPELINE_FILE = os.getenv(
    "AZUREML_PIPELINE_FILE", "pipelines/single-step-merge-job.yaml"
).strip()
RUN_JOB = os.getenv("RUN_AZUREML_JOB", "false").strip().lower() in {"1", "true", "yes"}

required = {
    "AZURE_SUBSCRIPTION_ID": SUBSCRIPTION_ID,
    "AZURE_RESOURCE_GROUP": RESOURCE_GROUP,
    "AZUREML_WORKSPACE_NAME": WORKSPACE_NAME,
    "AZUREML_COMPUTE_NAME": COMPUTE_NAME,
}
missing = [name for name, value in required.items() if not value]
if missing:
    raise ValueError("Missing .env values: " + ", ".join(missing))

pipeline_path = REPO_ROOT / PIPELINE_FILE
if not pipeline_path.is_file():
    raise FileNotFoundError(f"Pipeline YAML not found: {PIPELINE_FILE}")

credential = AzureCliCredential(tenant_id=TENANT_ID or None)
ml_client = MLClient(
    credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME
)

print(f"Pipeline:  {PIPELINE_FILE}")
print(f"Workspace: {WORKSPACE_NAME}")
print(f"Compute:   {COMPUTE_NAME}")
print(f"Submit:    {RUN_JOB}")

In [ ]:
job = load_job(pipeline_path)

if job.type == "pipeline":
    job.settings.default_compute = f"azureml:{COMPUTE_NAME}"
    if "automl_compute" in job.inputs:
        job.inputs["automl_compute"] = COMPUTE_NAME
else:
    job.compute = f"azureml:{COMPUTE_NAME}"

print(f"Loaded:  {type(job).__name__}")
print(f"Compute: {COMPUTE_NAME}")

In [ ]:
if RUN_JOB:
    submitted_job = ml_client.jobs.create_or_update(job)
    print(f"Submitted: {submitted_job.name}")

    ml_client.jobs.stream(submitted_job.name)
    final_job = ml_client.jobs.get(submitted_job.name)
    print(f"Status: {final_job.status}")

    if final_job.status != "Completed":
        raise RuntimeError(f"Job ended with status {final_job.status}")
else:
    print("Submission disabled. Set RUN_AZUREML_JOB=true in .env to run it.")